# Writing data to and reading data from a Database using Python (Supermarkets)

## Libraries and settings

In [ ]:
# Libraries
import os
import sqlite3
import fnmatch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Function to close a sqlite db-connection
def check_conn(conn):
     try:
        conn.cursor()
        return True
     except Exception as ex:
        return False

# Get current working directory
print(os.getcwd())

## Create sqlite data base

In [ ]:
# Create data base
conn = sqlite3.connect('supermarkets_database.db') 
cursor = conn.cursor()

# Show dbs in the directory
flist = fnmatch.filter(os.listdir('.'), '*.db')
for i in flist:
    print(i)

## Create SQL-table in the database

In [ ]:
cursor.execute('''CREATE TABLE IF NOT EXISTS supermarkets_table (Type VARCHAR(50),
                                                                 Id INT(20),
                                                                 Lat DECIMAL(10,7),
                                                                 Lon DECIMAL(10,7),
                                                                 Brand VARCHAR(100),
                                                                 Shop VARCHAR(100),
                                                                 City VARCHAR(100),
                                                                 Street VARCHAR(200),
                                                                 Housenumber VARCHAR(50),
                                                                 Postcode VARCHAR(20),
                                                                 OpeningHours VARCHAR(500))''')
# Confirm changes to the table
conn.commit()

## Read data from file to data frame

In [ ]:
df = pd.read_csv('supermarkets_data_prepared.csv',
                  sep=',', 
                  encoding='utf-8')
print(df.shape)
df.head(5)

## Write data to the SQL-table in data base

In [ ]:
df.to_sql(name = 'supermarkets_table',
          con = conn,
          index = False,
          if_exists = 'replace')

## Query the SQL-table

In [ ]:
# Query the SQL-table - all supermarkets
cursor.execute('''SELECT *
               FROM supermarkets_table''')

df_result = pd.DataFrame(cursor.fetchall(), 
                  columns=['Type','Id','Lat','Lon','Brand','Shop','City','Street','Housenumber','Postcode','OpeningHours'])    
print(f'Total number of supermarkets: {len(df_result)}')
df_result.head(10)

## Additional SQL-queries

### Query: Filter all supermarkets in the city of Winterthur

In [ ]:
# Query supermarkets in Winterthur
cursor.execute('''SELECT *
               FROM supermarkets_table
               WHERE city = 'Winterthur' ''')

df_winterthur = pd.DataFrame(cursor.fetchall(), 
                  columns=['Type','Id','Lat','Lon','Brand','Shop','City','Street','Housenumber','Postcode','OpeningHours'])    
print(f'Number of supermarkets in Winterthur: {len(df_winterthur)}')
df_winterthur

## Plot distribution of supermarket brands

In [ ]:
# Count supermarkets by brand
brand_counts = df_result['Brand'].value_counts().head(10)
brand_counts.plot.bar(color='#607c8e')
plt.title('Top 10 Supermarket Brands')
plt.xlabel('Brand')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

## Close db connection (if open)

In [ ]:
# Close db connection (if open)
try:
    if check_conn(conn):
        conn.close()
    else:
        pass
except:
    pass

# Status (True = open, False = closed)
print(check_conn(conn))

### Jupyter notebook --footer info-- (please always provide this at the end of each submitted notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')